In [15]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [16]:
# import master df
master_df = pd.read_csv('master_df.csv')

# calculate the
master_df['target_future_return'] = master_df['SPY_Close'].pct_change().shift(-1)

# create a new df called df_merge to take time, y_actual, y_pred's
df_merge = pd.DataFrame()
df_merge[['Date', 'y_actual']] = master_df[['Date', 'target_future_return']]

# import csv datasets of results from ML models
model_lin_reg = pd.read_csv('lin_reg_momentum_only_results.csv')
model_lin_reg = model_lin_reg.rename(columns = {'predicted_return': 'y_pred_lin_reg'})

model_multi_lin_reg = pd.read_csv('multi_lin_reg_with_momentum_and_VIX.csv')
model_multi_lin_reg = model_multi_lin_reg.rename(columns = {'predicted_return': 'y_pred_multi_lin_reg'})


model_random_forest = pd.read_csv('random_forest_results.csv')
model_random_forest = model_random_forest.rename(columns = {'predicted_return': 'y_pred_random_forest'})

# merge the df_merge with the model results
df_merge = pd.merge(
    left = df_merge,
    right = model_lin_reg[['Date', 'y_pred_lin_reg']],
    on = 'Date',
    how = 'left'
)
df_merge = pd.merge(
    left = df_merge,
    right = model_multi_lin_reg[['Date', 'y_pred_multi_lin_reg']],
    on = 'Date',
    how = 'left'
)
df_merge = pd.merge(
    left = df_merge,
    right = model_random_forest[['Date', 'y_pred_random_forest']],
    on = 'Date',
    how = 'left'
)

# make sure the Date is in datetime format
df_merge['Date'] = pd.to_datetime(df_merge['Date'], format='%Y-%m-%d')

df_merge.head()

,Date,y_actual,y_pred_lin_reg,y_pred_multi_lin_reg,y_pred_random_forest
0,2005-01-03,-0.012220,NaN,NaN,NaN
1,2005-01-04,-0.006900,NaN,NaN,NaN
2,2005-01-05,0.005085,NaN,NaN,NaN
3,2005-01-06,-0.001434,NaN,NaN,NaN
4,2005-01-07,0.004728,NaN,NaN,NaN


In [17]:
# import master df
master_df = pd.read_csv('master_df.csv')

# calculate the
master_df['target_future_return'] = master_df['SPY_Close'].pct_change().shift(-1)

# create a new df called df_merge to take time, y_actual, y_pred's
df_merge = pd.DataFrame()
df_merge[['Date', 'y_actual']] = master_df[['Date', 'target_future_return']]

# import csv datasets of results from ML models
model_lin_reg = pd.read_csv('lin_reg_momentum_only_results.csv')
model_lin_reg = model_lin_reg.rename(columns = {'predicted_return': 'y_pred_lin_reg'})

model_multi_lin_reg = pd.read_csv('multi_lin_reg_with_momentum_and_VIX.csv')
model_multi_lin_reg = model_multi_lin_reg.rename(columns = {'predicted_return': 'y_pred_multi_lin_reg'})


model_random_forest = pd.read_csv('random_forest_results.csv')
model_random_forest = model_random_forest.rename(columns = {'predicted_return': 'y_pred_random_forest'})

# merge the df_merge with the model results
df_merge = pd.merge(
    left = df_merge,
    right = model_lin_reg[['Date', 'y_pred_lin_reg']],
    on = 'Date',
    how = 'left'
)
df_merge = pd.merge(
    left = df_merge,
    right = model_multi_lin_reg[['Date', 'y_pred_multi_lin_reg']],
    on = 'Date',
    how = 'left'
)
df_merge = pd.merge(
    left = df_merge,
    right = model_random_forest[['Date', 'y_pred_random_forest']],
    on = 'Date',
    how = 'left'
)

In [18]:
# -----------------------------------------------------------
# 1️⃣ Create Volatility Regime Labels (based on VIX)
# -----------------------------------------------------------
# Convert Date to datetime and align
master_df['Date'] = pd.to_datetime(master_df['Date'])
df_merge['Date'] = pd.to_datetime(df_merge['Date'])

# Clean VIX and smooth slightly (optional)
master_df['VIX_SMA_5'] = master_df['VIX'].rolling(5).mean()

# Define thresholds (quantile-based gives more balance than median split)
low_th = master_df['VIX_SMA_5'].quantile(0.33)
high_th = master_df['VIX_SMA_5'].quantile(0.67)

def classify_vol(vix):
    if vix <= low_th:
        return 'Low Vol'
    elif vix <= high_th:
        return 'Medium Vol'
    else:
        return 'High Vol'

master_df['Vol_Regime'] = master_df['VIX_SMA_5'].apply(classify_vol)

# Merge regime labels into df_merge
df_merge = df_merge.merge(master_df[['Date', 'Vol_Regime']], on='Date', how='left')

# show the df_merge
df_merge.head()

,Date,y_actual,y_pred_lin_reg,y_pred_multi_lin_reg,y_pred_random_forest,Vol_Regime
0,2005-01-03,-0.012220,NaN,NaN,NaN,High Vol
1,2005-01-04,-0.006900,NaN,NaN,NaN,High Vol
2,2005-01-05,0.005085,NaN,NaN,NaN,High Vol
3,2005-01-06,-0.001434,NaN,NaN,NaN,High Vol
4,2005-01-07,0.004728,NaN,NaN,NaN,Low Vol


In [19]:
# -----------------------------------------------------------
# 2️⃣ Compute Model Errors
# -----------------------------------------------------------
# Signed errors (predicted - actual)
df_merge['error_lin_reg'] = df_merge['y_pred_lin_reg'] - df_merge['y_actual']
df_merge['error_multi_lin_reg'] = df_merge['y_pred_multi_lin_reg'] - df_merge['y_actual']
df_merge['error_random_forest'] = df_merge['y_pred_random_forest'] - df_merge['y_actual']

# Absolute errors (for performance magnitude)
df_merge['abs_error_lin_reg'] = df_merge['error_lin_reg'].abs()
df_merge['abs_error_multi_lin_reg'] = df_merge['error_multi_lin_reg'].abs()
df_merge['abs_error_random_forest'] = df_merge['error_random_forest'].abs()

# show the df_merge
df_merge.head(100)

,Date,y_actual,y_pred_lin_reg,y_pred_multi_lin_reg,y_pred_random_forest,Vol_Regime,error_lin_reg,error_multi_lin_reg,error_random_forest,abs_error_lin_reg,abs_error_multi_lin_reg,abs_error_random_forest
0,2005-01-03,-0.012220,NaN,NaN,NaN,High Vol,NaN,NaN,NaN,NaN,NaN,NaN
1,2005-01-04,-0.006900,NaN,NaN,NaN,High Vol,NaN,NaN,NaN,NaN,NaN,NaN
2,2005-01-05,0.005085,NaN,NaN,NaN,High Vol,NaN,NaN,NaN,NaN,NaN,NaN
3,2005-01-06,-0.001434,NaN,NaN,NaN,High Vol,NaN,NaN,NaN,NaN,NaN,NaN
4,2005-01-07,0.004728,NaN,NaN,NaN,Low Vol,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
95,2005-05-19,-0.001425,-0.000557,-0.000505,0.000435,Medium Vol,0.000867,0.000920,0.001859,0.000867,0.000920,0.001859
96,2005-05-20,0.005540,-0.000580,-0.000369,0.000477,Low Vol,-0.006120,-0.005910,-0.005063,0.006120,0.005910,0.005063
97,2005-05-23,-0.002337,-0.000429,-0.000483,0.000312,Low Vol,0.001909,0.001854,0.002650,0.001909,0.001854,0.002650
98,2005-05-24,-0.000752,-0.000070,-0.000022,0.000354,Low Vol,0.000682,0.000731,0.001106,0.000682,0.000731,0.001106


# Next Chart starts here

In [21]:
# --- Ensure all Date columns are datetime BEFORE merging ---

# Convert Date columns to datetime in every DataFrame
master_df['Date'] = pd.to_datetime(master_df['Date'])
model_lin_reg['Date'] = pd.to_datetime(model_lin_reg['Date'])
model_multi_lin_reg['Date'] = pd.to_datetime(model_multi_lin_reg['Date'])
model_random_forest['Date'] = pd.to_datetime(model_random_forest['Date'])
df_merge['Date'] = pd.to_datetime(df_merge['Date'])

# --- Extract and merge 'correct_direction' columns from model outputs ---

# Add model name suffixes for clarity
model_lin_reg = model_lin_reg.rename(columns={'correct_direction': 'correct_direction_lin_reg'})
model_multi_lin_reg = model_multi_lin_reg.rename(columns={'correct_direction': 'correct_direction_multi_lin_reg'})
model_random_forest = model_random_forest.rename(columns={'correct_direction': 'correct_direction_random_forest'})

# Merge these boolean columns into df_merge
df_merge = pd.merge(
    left=df_merge,
    right=model_lin_reg[['Date', 'correct_direction_lin_reg']],
    on='Date',
    how='left'
)

df_merge = pd.merge(
    left=df_merge,
    right=model_multi_lin_reg[['Date', 'correct_direction_multi_lin_reg']],
    on='Date',
    how='left'
)

df_merge = pd.merge(
    left=df_merge,
    right=model_random_forest[['Date', 'correct_direction_random_forest']],
    on='Date',
    how='left'
)

# Sanity check — ensure boolean dtype
for col in [
    'correct_direction_lin_reg',
    'correct_direction_multi_lin_reg',
    'correct_direction_random_forest'
]:
    df_merge[col] = df_merge[col].astype(bool)

# Verify results
print(df_merge[['Date', 'Vol_Regime',
                'correct_direction_lin_reg',
                'correct_direction_multi_lin_reg',
                'correct_direction_random_forest']].head())

print("\n✅ Date types verified:")
print(df_merge.dtypes[['Date']])


        Date Vol_Regime  correct_direction_lin_reg  \
0 2005-01-03   High Vol                       True   
1 2005-01-04   High Vol                       True   
2 2005-01-05   High Vol                       True   
3 2005-01-06   High Vol                       True   
4 2005-01-07    Low Vol                       True   

   correct_direction_multi_lin_reg  correct_direction_random_forest  
0                             True                             True  
1                             True                             True  
2                             True                             True  
3                             True                             True  
4                             True                             True  

✅ Date types verified:
Date    datetime64[ns]
dtype: object
